### Setup and Installation
This cell installs the necessary Python libraries for building a Retrieval-Augmented Generation (RAG) pipeline using LlamaIndex. It includes components for language models (LLMs), embeddings, retrievers (BM25), document readers, and asynchronous operations.

In [ ]:
!pip -q install -U \
  llama-index \
  llama-index-llms-openai \
  llama-index-embeddings-openai \
  llama-index-embeddings-huggingface \
  llama-index-llms-llama-cpp \
  llama-index-retrievers-bm25 \
  llama-index-readers-file \
  nest-asyncio

In [ ]:
import torch

# Check if GPU is available
if torch.cuda.is_available():
    print("GPU is available, using GPU.")
    device = "cuda"
else:
    print("GPU not available, using CPU.")
    device = "cpu"

!pip -q install -U sentence-transformers

Using `BAAI/bge-small-en-v1.5` as the local embedding model, which is a good balance between performance and size.

In [ ]:
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5",
    device=device
)

### Asynchronous Operations Fix
`nest_asyncio` is imported and `nest_asyncio.apply()` is called to allow nested execution of asyncio event loops. This is often necessary in environments like Jupyter or Colab where an event loop might already be running, preventing other asynchronous operations.

In [ ]:
import nest_asyncio
nest_asyncio.apply()

### API Key Configuration
This cell imports the `os` module to manage environment variables and `userdata` from `google.colab` to securely retrieve API keys. It then sets the OpenAI API key as an environment variable, which is used by LlamaIndex to authenticate with OpenAI services.

In [ ]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("APIKEY")

print("OpenAI key loaded:", bool(os.environ.get("OPENAI_API_KEY")))

### Create Knowledge Base Directory
`os.makedirs` is used to create a directory named `tech_kb`. The `exist_ok=True` argument ensures that if the directory already exists, no error is raised.

In [ ]:
os.makedirs("tech_kb", exist_ok=True)

### Define Knowledge Base Content
This dictionary, `docs`, stores the content for our knowledge base. Each key is a filename, and its value is a string containing textual information about RAG, hybrid search, reranking, query decomposition, and citations.

In [ ]:
docs = {
    "rag_basics.txt": """Retrieval-Augmented Generation, or RAG, combines retrieval with language generation.\nA typical RAG pipeline retrieves relevant chunks and uses them as context for the LLM.\nRAG reduces hallucination by grounding answers in external knowledge.\nHowever, vector-only retrieval may miss results when exact keywords matter or when the query has multiple parts.""",

    "hybrid_search.txt": """Hybrid retrieval combines dense vector retrieval and sparse BM25 retrieval.\nDense retrieval captures semantic similarity between query and text.\nBM25 captures keyword overlap and term importance.\nReciprocal Rank Fusion combines ranked lists from multiple retrievers to improve retrieval quality.""",

    "reranking_note.txt": """In production systems, a second-stage reranker may be used after retrieval.\nIn this lab, we use reciprocal rank fusion as the ranking strategy, so no separate reranker API is required.""",

    "query_decomposition.txt": """Query decomposition breaks one complex user question into smaller sub-queries.\nThis improves recall because each sub-query can retrieve useful evidence.\nFusion-based retrieval systems often combine query decomposition with multi-retriever search.""",

    "citations.txt": """Citations show which source documents support the generated answer.\nThey improve trust, transparency, and verification in a RAG system.\nLlamaIndex can return source nodes along with the final response."""
}

### Write Content to Files
This cell iterates through the `docs` dictionary. For each entry, it uses `textwrap.dedent` to remove common leading whitespace from multiline strings and then writes the content to a corresponding text file inside the `tech_kb` directory. This simulates having multiple knowledge base documents.

In [ ]:
import textwrap
for filename, content in docs.items():
    with open(os.path.join("tech_kb", filename), "w", encoding="utf-8") as f:
        f.write(textwrap.dedent(content).strip())

### Load Documents
`SimpleDirectoryReader` from LlamaIndex is used to load all text documents from the `tech_kb` directory. These documents will form the basis of our RAG system's knowledge.

In [ ]:
from llama_index.core import SimpleDirectoryReader
documents = SimpleDirectoryReader("tech_kb").load_data()
print("Documents loaded:", len(documents))

### Configure LlamaIndex Settings
This cell imports and configures core components of LlamaIndex:
- `OpenAI`: Sets the Language Model (LLM) to 'gpt-4o-mini' with a temperature of 0 for deterministic responses.
- `OpenAIEmbedding`: Sets the embedding model to 'text-embedding-3-small' for generating numerical representations of text.
- `SentenceSplitter`: Configures the node parser to split documents into chunks of 256 tokens with an overlap of 40 tokens. This helps in processing documents into manageable units for retrieval.

In [ ]:
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import Settings

Settings.node_parser = SentenceSplitter(chunk_size=256, chunk_overlap=40)

### Create Vector Store Index
`VectorStoreIndex.from_documents` creates an index from the loaded documents. This index uses the configured embedding model to convert document chunks into vectors, which allows for semantic similarity search.

In [ ]:
from llama_index.core import VectorStoreIndex

vector_index = VectorStoreIndex.from_documents(documents)
print("Vector index ready")

### Initialize Vector Retriever
This cell converts the `vector_index` into a `vector_retriever`. This retriever is responsible for fetching the top `similarity_top_k` (4 in this case) most semantically similar document chunks based on a query.

In [ ]:
vector_retriever = vector_index.as_retriever(similarity_top_k=4)

### Initialize BM25 Retriever
This cell sets up a `BM25Retriever`, which is a sparse retrieval method that focuses on keyword matching and term frequency. It first gets nodes (chunks) from the documents using the configured `SentenceSplitter` and then initializes the BM25 retriever with these nodes, also configured to return the top 4 results.

In [ ]:
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core import Settings

nodes = Settings.node_parser.get_nodes_from_documents(documents)
bm25_retriever = BM25Retriever.from_defaults(nodes=nodes, similarity_top_k=4)

### Create Query Fusion Retriever
`QueryFusionRetriever` combines the results from multiple retrievers (in this case, the `vector_retriever` and `bm25_retriever`). It uses `reciprocal_rerank` mode to combine and re-rank the results, aiming to improve retrieval quality by leveraging both semantic and keyword-based search. `num_queries` is set to 4 to generate additional sub-queries for broader retrieval.

In [ ]:
from llama_index.core.retrievers import QueryFusionRetriever

fusion_retriever = QueryFusionRetriever(
    [vector_retriever, bm25_retriever],
    similarity_top_k=6,
    num_queries=4,
    mode="reciprocal_rerank",
    use_async=True,
    verbose=True,
)

print("Hybrid fusion retriever created")

### Configure Response Synthesizer
`get_response_synthesizer` creates an object responsible for generating the final answer from the retrieved document chunks. The `response_mode='compact'` setting instructs it to use a compact approach for synthesizing the response, summarizing information efficiently.

In [ ]:
from llama_index.core.response_synthesizers import get_response_synthesizer

response_synthesizer = get_response_synthesizer(response_mode="compact")

### Create Retriever Query Engine
This cell initializes the `RetrieverQueryEngine`, which is the core component of our RAG pipeline. It takes the `fusion_retriever` to retrieve relevant context and the `response_synthesizer` to generate an answer based on that context. This engine orchestrates the entire query process.

In [ ]:
from llama_index.core.query_engine import RetrieverQueryEngine

query_engine = RetrieverQueryEngine(
    retriever=fusion_retriever,
    response_synthesizer=response_synthesizer,
)

print("Advanced RAG pipeline ready")

### Define the Query
This markdown cell simply defines the `query` string that will be passed to the RAG pipeline. The query asks for an explanation of several advanced RAG concepts and the importance of citations.

In [ ]:
query = """
Explain how query decomposition, hybrid retrieval, and reciprocal rank fusion
work together in an advanced RAG pipeline. Also mention why citations matter.
"""

### Execute the Query with Local LLM
This cell executes the defined `query` using the `query_engine` with a local LlamaCPP model. The `query_engine` will perform retrieval, synthesis, and generate a response based on the knowledge base.

In [ ]:
!pip -q install llama-cpp-python --force-reinstall --no-deps

Download a local GGUF model. Using `NousResearch/Nous-Hermes-2-Mistral-7B-DPO-GGUF` as an example. This model offers a good balance of performance and size for local execution.

In [ ]:
from huggingface_hub import hf_hub_download
import os

model_name = "NousResearch/Nous-Hermes-2-Mistral-7B-DPO-GGUF"
model_file = "Nous-Hermes-2-Mistral-7B-DPO.Q4_K_M.gguf"

model_path = hf_hub_download(repo_id=model_name, filename=model_file)
print(f"Downloaded model to: {model_path}")

Configure LlamaIndex to use this local LLM by importing `LlamaCPP` and setting `Settings.llm`.

In [ ]:
from llama_index.llms.llama_cpp import LlamaCPP

llm = LlamaCPP(
    model_path=model_path,
    temperature=0.1,
    max_new_tokens=256,
    context_window=3900,
    generate_kwargs={},
    model_kwargs={"n_gpu_layers": -1},
    verbose=True,
)

Settings.llm = llm
print("LlamaIndex configured to use local LlamaCPP model.")

### Execute Query and Display Results
Execute the query using the RAG pipeline with the local LLM and display the response with source citations.

In [ ]:
# Execute the query
response = query_engine.query(query)

print("\n" + "=" * 80)
print("FINAL ANSWER")
print("=" * 80)
print(response)

print("\n" + "=" * 80)
print("SOURCE CITATIONS")
print("=" * 80)
for i, source in enumerate(response.source_nodes, 1):
    print(f"\nSource {i}")
    print("File:", source.metadata.get("file_name"))
    print("Score:", source.score)
    print("Snippet:", source.text[:200])

print("\n" + "=" * 80)
print("RAG query complete with local LLM.")
print("=" * 80)